# Modules, Packages, Environments, Type Hints and Validation

Notebook for **Chapter 9** of the *python* study document. Run the cells top to bottom. Read the full explanation on the site: https://dadipavan.github.io/Learning/study/python/ch09.html

Tip: change a number, predict the output, then run the cell again.

## 9.1 Modules, Packages and__main__

In [ ]:
# shop/__init__.py
"""shop package."""
from .pricing import total_with_tax
__all__ = ["total_with_tax"]
__version__ = "0.1.0"
print("shop/__init__ ran")

In [ ]:
# shop/pricing.py
from .utils import round2
TAX = 0.18
def total_with_tax(amount):
    return round2(amount * (1 + TAX))
print("pricing imported, __name__ =", __name__)

In [ ]:
# shop/utils.py
def round2(x):
    return round(x, 2)

In [ ]:
# shop/__main__.py
from shop import total_with_tax
import sys
def main():
    amount = float(sys.argv[1]) if len(sys.argv) > 1 else 100.0
    print("total:", total_with_tax(amount))
if __name__ == "__main__":
    main()

## 9.2 Circular Imports and How to Avoid Them

In [ ]:
# a.py
from b import fb
def fa(): return "fa"

In [ ]:
# b.py
from a import fa
def fb(): return "fb"

In [ ]:
# b.py
def fb():
    from a import fa      # deferred import
    return "fb+" + fa()

## 9.3 Virtual Environments and Package Managers

## 9.4pyproject.toml, Version Pinning and Supply-Chain Awareness

In [ ]:
import tomllib
text = """
[project]
name = "shop"
version = "0.1.0"
requires-python = ">=3.10"
dependencies = ["requests>=2.31"]
"""
print(tomllib.loads(text))

{'project': {'name': 'shop', 'version': '0.1.0', 'requires-python': '>=3.10', 'dependencies': ['requests>=2.31']}}


In [ ]:
import importlib.metadata as m
print(sorted(d.metadata['Name'] for d in m.distributions())[:5])

['Jinja2', 'MarkupSafe', 'PyYAML', 'Pygments', 'Send2Trash']


## 9.5 Type Hints and mypy

In [ ]:
from typing import Optional, Union, Callable, TypeVar, Protocol, Iterable

def mean(xs: list[float]) -> float:
    return sum(xs) / len(xs)

def find_user(users: dict[int, str], uid: int) -> Optional[str]:
    return users.get(uid)

def parse(v: Union[int, str]) -> int:
    return int(v)
def parse2(v: int | str) -> int:    # modern spelling
    return int(v)

def apply(f: Callable[[int], int], xs: Iterable[int]) -> list[int]:
    return [f(x) for x in xs]

T = TypeVar("T")
def first(xs: list[T]) -> T:
    return xs[0]
def first_new[T](xs: list[T]) -> T:   # Python 3.12+ syntax
    return xs[0]

class HasName(Protocol):
    name: str
def greet(x: HasName) -> str: return "hi " + x.name

print(mean([1,2,3]), find_user({1:"a"}, 2), parse("7"), parse2(8))
print(apply(lambda n: n*n, [1,2,3]), first(["x","y"]), first_new([10,20]))

2.0 None 7 8
[1, 4, 9] x 10


In [ ]:
# This cell is meant to fail: read the error message, that is the lesson.
try:
    print(mean("abc"))
except TypeError as e: print("TypeError:", e)
from typing import get_type_hints
print(get_type_hints(mean))

TypeError: unsupported operand type(s) for +: 'int' and 'str'
{'xs': list[float], 'return': <class 'float'>}


## 9.6 Data Validation: By Hand and with pydantic

In [ ]:
from dataclasses import dataclass, fields

class ValidationError(Exception):
    pass

@dataclass
class Order:
    order_id: int
    email: str
    amount: float
    def __post_init__(self):
        errors = []
        for f in fields(self):
            v = getattr(self, f.name)
            try:
                setattr(self, f.name, f.type(v))   # coerce like pydantic lax mode
            except (TypeError, ValueError):
                errors.append(f"{f.name}: cannot convert {v!r} to {f.type.__name__}")
        if not errors and "@" not in self.email: errors.append("email: missing @")
        if not errors and self.amount <= 0: errors.append("amount: must be > 0")
        if errors: raise ValidationError("; ".join(errors))

print(Order("42", "a@b.com", "99.5"))
for bad in [("x", "a@b.com", 5), (1, "nomail", 5), (1, "a@b.com", -3)]:
    try: Order(*bad)
    except ValidationError as e: print("rejected:", e)

Order(order_id=42, email='a@b.com', amount=99.5)
rejected: order_id: cannot convert 'x' to int
rejected: email: missing @
rejected: amount: must be > 0


In [ ]:
from pydantic import BaseModel, EmailStr, Field, ValidationError

class Order(BaseModel):
    order_id: int
    email: EmailStr          # needs: pip install "pydantic[email]"
    amount: float = Field(gt=0)

try:
    Order(order_id="42", email="a@b.com", amount=-3)
except ValidationError as e:
    print(e)                 # lists every failing field

## 9.7 Jupyter Notebooks vs Scripts,nbconvertand Reproducibility

In [ ]:
import json
nb = {"cells":[{"cell_type":"code","source":["print(1+1)\n"],"outputs":[],"execution_count":None,"metadata":{}}],"metadata":{},"nbformat":4,"nbformat_minor":5}
open("/tmp/c9/x.ipynb","w").write(json.dumps(nb))
d = json.load(open("/tmp/c9/x.ipynb"))
for c in d["cells"]:
    print("".join(c["source"]), end="")
    exec("".join(c["source"]))

print(1+1)
2


## 9.8 Typing in Depth, Import Order and Packaging Layout

Section 9.5 gave you the basics: annotations and a type checker. Real code needs more precise promises: "this dictionary has exactly these keys", "this string is one of three values", "this decorator keeps the signature of the function it wraps". This section adds those tools. A key fact first: **Python itself ignores annotations at run time**. A separate program, a type checker such as mypy or pyright, reads them. Every code block below really ran, so its output is real. Checker messages are labelled *not run here* because mypy is not installed in this environment.

In [ ]:
from typing import TypedDict, NotRequired
import json

class User(TypedDict):
    id: int
    name: str
    email: NotRequired[str]

payload = json.loads('{"id": 7, "name": "Ada"}')
u: User = payload
print(u["name"], type(u).__name__, "email" in u)
print(sorted(User.__required_keys__), sorted(User.__optional_keys__))

Ada dict False
['id', 'name'] ['email']


In [ ]:
from typing import Literal, Final, Self, Protocol

Level = Literal["debug", "info", "error"]
def set_level(level: Level) -> str:
    return level.upper()
print(set_level("info"))
print(set_level("verbose"))   # a checker rejects this line; Python runs it

MAX_TRIES: Final = 3
print(MAX_TRIES)

class Builder:
    def __init__(self) -> None:
        self.parts: list[str] = []
    def add(self, p: str) -> Self:
        self.parts.append(p)
        return self
class SqlBuilder(Builder):
    def build(self) -> str:
        return " ".join(self.parts)
print(SqlBuilder().add("SELECT 1").add("FROM t").build())

class SupportsClose(Protocol):
    def close(self) -> None: ...
class Conn:
    def close(self) -> None:
        print("closed")
def shutdown(x: SupportsClose) -> None:
    x.close()
shutdown(Conn())

INFO
VERBOSE
3
SELECT 1 FROM t
closed


In [ ]:
from typing import overload

@overload
def parse(v: str) -> int: ...
@overload
def parse(v: list[str]) -> list[int]: ...
def parse(v):
    return int(v) if isinstance(v, str) else [int(i) for i in v]
print(parse("5"), parse(["1", "2"]))

5 [1, 2]


In [ ]:
import functools
from collections.abc import Callable

def retry[**P, R](times: int) -> Callable[[Callable[P, R]], Callable[P, R]]:
    def deco(fn: Callable[P, R]) -> Callable[P, R]:
        @functools.wraps(fn)
        def wrapper(*args: P.args, **kwargs: P.kwargs) -> R:
            last: Exception | None = None
            for attempt in range(1, times + 1):
                try:
                    return fn(*args, **kwargs)
                except ValueError as e:
                    last = e
                    print(f"attempt {attempt} failed")
            assert last is not None
            raise last
        return wrapper
    return deco

calls = {"n": 0}
@retry(3)
def flaky(x: int, *, scale: int = 2) -> int:
    calls["n"] += 1
    if calls["n"] < 3:
        raise ValueError("boom")
    return x * scale
print(flaky(5, scale=10), flaky.__name__)

attempt 1 failed
attempt 2 failed
50 flaky


In [ ]:
from typing import TypedDict, NotRequired

class Order(TypedDict):
    sku: str
    qty: int
    note: NotRequired[str]

def total_units(orders: list[Order]) -> int:
    return sum(o["qty"] for o in orders)

print(total_units([{"sku": "A1", "qty": 3}, {"sku": "B2", "qty": 4, "note": "gift"}]))

7


In [ ]:
def f(x: int) -> int:
    return x * 2
print(f("ab"))

abab


## Chapter recap

## Practice set

Problem 9.8.1 (easy) What is `__name__` inside `shop/pricing.py` when it is imported by `shop/__init__.py`?

## Prompt pack

---
Done? Do the **Practice set** at the end of the chapter, then clear your Review cards on the site.

https://dadipavan.github.io/Learning/study/python/ch09.html